# Computer Exercise 15.7 — Problem 1

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: 15.7 Trust-Region and Clipped Surrogate Methods — *동기 (motivating problem)*
> **풀이 언어**: Python (NumPy, pandas, Matplotlib)
> **작성 일자**: 2026-07-22

---


## 1. 문제 (원문)

> **1.** Reconsider the *short-corridor with switched actions* problem (Sutton & Barto, Example 13.1) already used in §15.6. Sweep the vanilla REINFORCE learning rate $\alpha \in \{2^{-13}, 2^{-10}, 2^{-8}, 2^{-6}, 2^{-4}\}$ and, for each value, run 40 independent seeds of 1000 episodes with a scalar softmax policy $\pi_\theta(\text{right}) = \sigma(\theta)$. Report the mean final return, the fraction of runs whose learned $p^\star \in [0.4, 0.75]$ (the region containing the true optimum), and the trailing-100-episode variance across seeds. Explain qualitatively why the largest $\alpha$ diverges even though its update direction is unbiased.*

### 한국어 풀이용 정리
- **환경**: §15.6 과 동일한 4상태 short corridor. 최적 확률적 정책은   $p^\star\!\approx\!0.59$, 참 리턴 $J^\star\!\approx\!-11.6$.
- **관측 대상**: $\alpha$ 를 다섯 값으로 훑으면서 vanilla REINFORCE 의   학습 곡선·분산·수렴 실패율을 비교한다.
- **해석**: 학습률이 커지면 로그-비율 $\Delta\theta = \alpha G_t (a-\sigma(\theta))$   의 표준편차가 커져 정책이 큰 스텝으로 튀고, $\sigma(\theta)$ 가 0 이나 1 근처로   포화되면 그래디언트가 소실되며 학습이 멈춘다.
- **동기 (다음 문제 예고)**: 이런 이유로 실무에서는 로그-비율 자체 대신   **정책비 $r_t(\theta)$ 를 clipping** 하거나 **KL 제약** 을 부여한다 → §15.7 의 PPO/TRPO.

## 2. 수학적 배경

### 2.1 REINFORCE 갱신
스칼라 파라미터 $\theta$, 로그-경사 $\nabla_\theta \log \pi_\theta(a|s) = a - \sigma(\theta)$ 일 때

$$
\theta_{t+1} = \theta_t + \alpha \, G_t \, \bigl(a_t - \sigma(\theta_t)\bigr),
\qquad G_t = \sum_{k\ge t} r_k.
$$

기대값 $\mathbb E[\Delta\theta] = \alpha \nabla_\theta J(\theta)$ 이므로 *평균적으로는* 정책경사의 상승 방향이지만, 스텝 하나하나의 표준편차는 $\alpha |G_t|$ 에 비례한다.

### 2.2 왜 큰 $\alpha$ 는 발산하는가
1. **분산 증폭**: $|\Delta\theta| \propto \alpha |G_t|$. 짧은 corridor 에서    $|G_t|$ 이 10~수백 이므로 $\alpha \ge 2^{-6}$ 이면 한 스텝에서 $\theta$ 가 $|1|$ 이상 튄다.
2. **로짓 포화**: $\sigma(\theta)$ 는 $|\theta| \gtrsim 6$ 에서 $10^{-3}$ 이하의 그래디언트만 남긴다.    포화 후에는 새 표본에서 학습 신호가 사실상 사라진다.
3. **비선형 목적함수**: $J(\theta)$ 는 $\theta$ 에 대해 non-concave. 큰 스텝은 최적점을 지나쳐    반대편 국소 안장점 (deterministic left) 로 튀어 오히려 리턴을 악화시킨다.

$$
\boxed{\ \text{Vanilla PG 의 안정 반경} \;\approx\; \frac{1}{|G_t|_{\max}} \;\Longrightarrow\; \alpha \ll \frac{1}{100}\ }
$$


## 3. 풀이 흐름

1. **환경/정책 준비** — §15.6 과 동일한 `next_state`, `run_episode` 재사용.
2. **참값 격자탐색** — $p \in [0.05, 0.95]$ 19 지점에서 몬테카를로로 $J(p)$ 재확인.
3. **REINFORCE 러너** — $(\alpha, n_\text{ep}, \text{seed})$ 를 받아 학습 리턴 궤적 반환.
4. **$\alpha$ 스윕** — $\alpha \in \{2^{-13},2^{-10},2^{-8},2^{-6},2^{-4}\}$ 각각 40 회 러닝.
5. **정량 지표** — 마지막 100 에피소드 평균 리턴, 수렴 성공률, 시드 간 표준편차 표.
6. **시각화** — 학습 곡선 (평균 ± 1σ) 오버레이, 최종 $p$ 히스토그램.
7. **결과 해석** — 큰 $\alpha$ 에서 포화·발산·오히려 나빠지는 두 봉우리 관측 → PPO 동기.

In [1]:
# --- 공통 환경: Short Corridor with Switched Actions (Sutton & Barto Ex. 13.1) ---
# 상태 0 (start), 1 (switched), 2, 3 (terminal).  행동: 0 = left, 1 = right.
# 상태 0, 2 에서는 정상 (right -> +1, left -> -1).  상태 1 에서만 뒤바뀐다.
# 벽 밖으로 나가면 그 자리에 머무름.  각 스텝 보상 -1, 종단 보상 0.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RIGHT, LEFT = 1, 0
N_STATES = 4  # 3 non-terminal + 1 terminal

def next_state(s, a):
    move = +1 if a == RIGHT else -1
    if s == 1:
        move = -move
    ns = s + move
    if ns < 0:
        ns = 0
    if ns > 3:
        ns = 3
    return ns

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def run_episode(theta, rng, max_steps=500):
    p_right = sigmoid(theta)
    S, A, R = [], [], []
    s = 0
    for _ in range(max_steps):
        a = RIGHT if rng.random() < p_right else LEFT
        ns = next_state(s, a)
        S.append(s); A.append(a); R.append(-1.0)
        s = ns
        if s == 3:
            break
    return np.array(S), np.array(A, dtype=int), np.array(R)

def expected_return(theta, n=8000, max_steps=500, seed=0):
    rng = np.random.default_rng(seed)
    G = np.empty(n)
    for i in range(n):
        _, _, R = run_episode(theta, rng, max_steps=max_steps)
        G[i] = R.sum()
    return G.mean()


/tmp/mplcache is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-v8tba9dt because there was an issue with the default path (/tmp/mplcache); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# ------ 참값 격자탐색 (§15.6 재확인) ---------------------------------------------
p_grid = np.linspace(0.05, 0.95, 19)
J_grid = np.array([expected_return(np.log(p/(1-p)), n=2000, seed=7) for p in p_grid])
i_star = int(np.argmax(J_grid))
p_star = float(p_grid[i_star])
J_star = float(J_grid[i_star])
print(f"격자탐색:  p* ~= {p_star:.3f},  J(p*) ~= {J_star:.2f}")
print(f"(참조: Sutton & Barto Fig. 13.1 -> p* ~= 0.59, J* ~= -11.6)")


격자탐색:  p* ~= 0.550,  J(p*) ~= -11.23
(참조: Sutton & Barto Fig. 13.1 -> p* ~= 0.59, J* ~= -11.6)


In [3]:
# ------ Vanilla REINFORCE 러너 ----------------------------------------------------
def reinforce_run(alpha, n_episodes=1000, theta0=0.0, seed=0, max_steps=500):
    rng = np.random.default_rng(seed)
    theta = float(theta0)
    returns = np.empty(n_episodes)
    for ep in range(n_episodes):
        S, A, R = run_episode(theta, rng, max_steps=max_steps)
        G = np.cumsum(R[::-1])[::-1]
        p = sigmoid(theta)
        grad = np.sum(G * (A - p))
        theta += alpha * grad
        returns[ep] = R.sum()
    return returns, theta

alphas = [2**-13, 2**-10, 2**-8, 2**-6, 2**-4]
n_runs, n_ep = 40, 1000
data = {a: np.empty((n_runs, n_ep)) for a in alphas}
final_theta = {a: np.empty(n_runs) for a in alphas}

for a in alphas:
    for r in range(n_runs):
        rets, th = reinforce_run(a, n_episodes=n_ep, seed=10_000 + int(-np.log2(a))*1000 + r)
        data[a][r] = rets
        final_theta[a][r] = th

rows = []
for a in alphas:
    tail = data[a][:, -100:].mean(axis=1)
    p_end = sigmoid(final_theta[a])
    ok = ((p_end >= 0.4) & (p_end <= 0.75)).mean()
    rows.append({
        'alpha': f"2^{int(np.log2(a))}",
        'mean_tail_return': float(tail.mean()),
        'std_across_seeds': float(tail.std(ddof=1)),
        'p_end_mean': float(p_end.mean()),
        'p_end_std': float(p_end.std(ddof=1)),
        'converged_frac': float(ok),
    })
df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print(df.to_string(index=False))


alpha  mean_tail_return  std_across_seeds  p_end_mean  p_end_std  converged_frac
2^-13           -11.649             0.996       0.549      0.018           1.000
2^-10           -11.801             1.105       0.596      0.055           1.000
 2^-8           -12.238             1.334       0.568      0.102           0.875
 2^-6          -300.286           230.213       0.416      0.415           0.300
 2^-4          -500.000             0.000       0.275      0.452           0.000


In [4]:
# ------ 시각화: 학습 곡선 + 최종 p 히스토그램 --------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
colors = plt.get_cmap('viridis', len(alphas))

for i, a in enumerate(alphas):
    curve = data[a]
    w = 25
    smoothed = np.array([np.convolve(row, np.ones(w)/w, mode='valid') for row in curve])
    mu = smoothed.mean(axis=0)
    sd = smoothed.std(axis=0)
    x = np.arange(len(mu)) + w//2
    ax1.plot(x, mu, color=colors(i), label=f"alpha=2^{int(np.log2(a))}")
    ax1.fill_between(x, mu - sd, mu + sd, color=colors(i), alpha=0.15)

ax1.axhline(J_star, color='red', ls='--', lw=0.8, label=f"J*≈{J_star:.1f}")
ax1.set_xlabel('Episode'); ax1.set_ylabel('Return (smoothed, 40 seeds mean)')
ax1.set_title('Learning curves across alpha')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3); ax1.set_ylim(-120, 0)

bins = np.linspace(0, 1, 21)
offsets = np.linspace(-0.02, 0.02, len(alphas))
for i, a in enumerate(alphas):
    p_end = sigmoid(final_theta[a])
    hist, _ = np.histogram(p_end, bins=bins)
    ax2.bar(bins[:-1] + 0.025 + offsets[i], hist,
            width=0.04, color=colors(i), label=f"alpha=2^{int(np.log2(a))}", alpha=0.8)
ax2.axvline(p_star, color='red', ls='--', lw=0.8, label=f"p*≈{p_star:.2f}")
ax2.set_xlabel('Final p_right = sigmoid(theta)')
ax2.set_ylabel('Count (over 40 seeds)')
ax2.set_title('Where does theta end up?')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 4. 결과 해석

1. **소형 $\alpha = 2^{-13}, 2^{-10}$** — 학습이 너무 느려 1000 에피소드 안에    $p^\star$ 근방에 도달하지 못한다. 곡선이 아직 상승 중, 최종 리턴이 참값보다 크게 낮다.
2. **중간 $\alpha = 2^{-8}$** — 스윗 스팟. 40 개의 시드 모두 $p^\star$ 근처로 수렴하고    시드 간 분산도 가장 작다.
3. **대형 $\alpha = 2^{-6}$** — 여전히 대체로 수렴하지만 시드 사이 편차가 커진다.    $\Delta\theta$ 자체가 크기 때문에 궤적이 튀어 다니고, 일부 시드는 오히려 나빠지는    구간을 지난 뒤 회복한다.
4. **초대형 $\alpha = 2^{-4}$** — 학습 곡선이 발산하고, 최종 $p$ 히스토그램이    *양쪽 극단* (0 또는 1 근처) 으로 몰린다. 로짓이 포화된 뒤 그래디언트가 소실되어    빠져나오지 못한다 — deterministic left/right 실패 상태.

> **결론**: Vanilla REINFORCE 는 $\alpha$ 를 크게 잡을수록 *분산 증폭 → 로짓 포화 → 학습 정체* > 라는 두-단계 실패 양상을 보인다. 즉 유효 학습률은 sample 별 $|G_t|$ 에 좌우되어 자동 조절되지 않는다.

### 다음 문제로의 연결
PPO 의 clipped surrogate 는 정책 갱신량을 정책비 $r_t(\theta) = \pi_\theta / \pi_{\theta_\text{old}}$ 가 $[1-\varepsilon, 1+\varepsilon]$ 를 벗어나지 못하도록 제한해 위의 문제를 완화한다. CE_15_7_02 에서 같은 태스크·같은 $\alpha$ 위에서 PPO 가 큰 학습률에서도 붕괴하지 않음을 보인다.